In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

# Configure the necessary Python module imports for dashboard components
from dash import Dash, dcc, html, Input, Output, State
import plotly.express as px
from dash import dash_table

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import base64
import dash_leaflet as dl



#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from aac_crud import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "aacuser"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# --- Test the CRUD connection
try:
    test_data = db.read({})
    print(type(test_data))
    print(len(test_data))
    print(test_data[0] if test_data else "No data found.") 
except Exception as e:
        print("Error:", e)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)
print("Total records:", len(df))
print("Columns:", list(df.columns))

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
import base64
image_filename = "Grazioso Salvare Logo.png" # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))
# Define rescue categories
rescue_types = [
    "Water Rescue",
    "Mountain or Wilderness Rescue",
    "Disaster or Individual Tracking",
    "Reset"
]

# App layout
app.layout = html.Div([
    html.Center(html.H1("Grazioso Salvare Dashboard")),
    html.Center(html.H4("Created by Benjamin Vallejo")), # Unique ID
    html.Hr(),
    
    # --- Filter Control---
    html.Label("Select Rescue Type:"),
    dcc.RadioItems(
        id='rescue-type-filter',
        options=[{'label': i, 'value': i} for i in rescue_types],
        value='Reset',
        inline=True
    ),
    
    html.Hr(),

#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 
# ---- Data Table

    
    dash_table.DataTable(
        id='datatable-id',
        columns=[{'name': i, 'id': i, 'selectable': True} for i in df.columns],
        row_selectable='single',
        selected_rows=[],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        style_table={'overflowX': 'auto'}
        
    ),
    
    html.Br(),
    
    # --- Charts
    html.Div([
        html.Div([
            html.H3("Geolocation Chart"),
            dcc.Graph(id='map-id')
        ], style={'width': '48%', 'display': 'inline-block'}),
        
        html.Div([
            html.H3("Breed Distribution"),
            dcc.Graph(id='graph-id')
        ], style={'width': '48%', 'display': 'inline-block'})
    ])
])
#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])

def update_dashboard(filter_type):
    #Define breed filters for each rescue
    if filter_type == "Water Rescue":
        breeds = ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]
        dff = df[df["breed"].isin(breeds)]
    elif filter_type == "Mountain or Wilderness Rescue":
        breeds = ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]
        dff = df[df["breed"].isin(breeds)]
    elif filter_type == "Disaster or Individual Tracking":
        breeds = ["German Shepherd", "Golder Retriever", "Bloodhound", "Rottweiler"]
        dff = df[df["breed"].isin(breeds)]
    else:
        dff = df # Reset: show all data
        
    # Return the filtered records to the DataTable
    return dff.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    dff = pd.DataFrame(viewData)

    if dff.empty or "breed" not in dff.columns:
        return [
            html.H4("No data available to display.", style={'color': 'red'})]

    fig = px.pie(dff, names='breed', title='Breed Distribution', hole=0.3)

    return [dcc.Graph(figure=fig)]
    
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')])

def update_map(viewData, index):  
    if viewData is None:
        return [dl.Map(style={'width': '1000px', 'height': '500px'})]
    
    dff = pd.DataFrame(viewData)
    
    if index is None or len(index) ==0:
        row = 0
    else:
        row = index[0]
        
    if 'location_lat' not in dff.columns or 'location_long' not in dff.columns:
        return [html.H4("No location data available.", style={'color': 'red'})]
    
    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']
    animal_name = dff.iloc[row]['name']
    #Plot location Austin
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75, -97.48], zoom=10,
               children=[
                   dl.TileLayer(id="base-layer-id"),
                   dl.Marker(
                       position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                       children=[
                           dl.Tooltip(dff.iloc[row, 4]),
                           dl.Popup([
                               html.H1("Animal Name"),
                               html.P(dff.iloc[row, 9])
                           ])
                       ]
                   )
               ]
              )
    ]
 
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode='inline')

 Found 19940 record(s) matching your query.
<class 'list'>
19940
{'rec_num': 1, 'age_upon_outcome': '3 years', 'animal_id': 'A746874', 'animal_type': 'Cat', 'breed': 'Domestic Shorthair Mix', 'color': 'Black/White', 'date_of_birth': '2014-04-10', 'datetime': '2017-04-11 09:00:00', 'monthyear': '2017-04-11T09:00:00', 'name': '', 'outcome_subtype': 'SCRP', 'outcome_type': 'Transfer', 'sex_upon_outcome': 'Neutered Male', 'location_lat': 30.5066578739455, 'location_long': -97.3408780722188, 'age_upon_outcome_in_weeks': 156.767857142857}
 Found 19940 record(s) matching your query.
Total records: 19940
Columns: ['rec_num', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'name', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks']


---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
TypeError: 'NoneType' object is not iterable

